In [7]:
#This script takes two .txt Bible files and the plg_verses.csv as input and returns aligned ENG+LANG files with all matching verse numbers

In [8]:
from pathlib import Path

def load_bible_txt_as_dict(path: str | Path) -> dict[str, str]:
    """
    Parse a monolingual Bible .txt file:
    - Skips metadata lines starting with '#'
    - Expects: '<VERSE_ID><whitespace><TEXT...>'
    Returns a dict mapping verse_id -> text.
    """
    path = Path(path)
    verses: dict[str, str] = {}

    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            # Split once: supports tab or space delimiters
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue  # why: ignore malformed lines quietly
            vid, text = parts[0], parts[1].strip()
            if vid in verses:
                # why: last one wins; avoids duplication noise if file has repeats
                pass
            verses[vid] = text

    return verses

# Example usage — set your file paths
eng_path = "eng-x-bible-newsimplified.txt"      # e.g., ENG file
other_path = "kat-x-bible.txt"     # e.g., DEU file

eng_map = load_bible_txt_as_dict(eng_path)
other_map = load_bible_txt_as_dict(other_path)

print(f"ENG verses:  {len(eng_map)}")
print(f"OTHER verses:{len(other_map)}")
# Quick spot-check:
for k in list(eng_map.keys())[:5]:
    print(k, "→", eng_map[k][:60], "...")

ENG verses:  31035
OTHER verses:4883
01001001 → In the beginning God created the heavens and the earth . ...
01001002 → The earth was without form and empty . Darkness covered the  ...
01001003 → Then God said : » Let there be light ! « And there was light ...
01001004 → God saw that the light was good . God separated the light fr ...
01001005 → God called the light day . He called the darkness night . Th ...


In [9]:
import pandas as pd
from pathlib import Path
import csv

# === Step 5: Build bilingual CSV of matching perfect-like verses ===

# Input files
plg_path = Path("plg_verses.csv")  # from Step 3
eng_path = Path("eng-x-bible-newsimplified.txt")
other_path = Path("kat-x-bible.txt")

# Language codes
lang1 = "ENG"
lang2 = "KAT"

# --- Load helper from previous cell (should already be defined) ---
# load_bible_txt_as_dict()

eng_map = load_bible_txt_as_dict(eng_path)
other_map = load_bible_txt_as_dict(other_path)

# --- Load list of verse IDs with perfect-like constructions ---
plg_ids = set(pd.read_csv(plg_path)["verse_id"].astype(str))

print(f"Loaded {len(plg_ids)} verse IDs from {plg_path.name}")

# --- Keep only verse IDs that appear in BOTH files ---
common_ids = [vid for vid in plg_ids if vid in eng_map and vid in other_map]
print(f"✅ {len(common_ids)} verses found in both ENG and {lang2}")

# --- Build DataFrame ---
records = []
for vid in common_ids:
    records.append({"verse_id": vid, "language": lang1, "text": eng_map[vid]})
    records.append({"verse_id": vid, "language": lang2, "text": other_map[vid]})

df_out = pd.DataFrame(records).sort_values(["verse_id", "language"]).reset_index(drop=True)

# --- Save output ---
output_path = Path(f"bible_{lang1.lower()}_{lang2.lower()}_perfects.csv")
df_out.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")

print(f"✅ Done! Created bilingual CSV: {output_path}")
print(f"Rows: {len(df_out)}  (≈ {len(common_ids)} verses × 2)")
display(df_out.head(6))

Loaded 1953 verse IDs from plg_verses.csv
✅ 220 verses found in both ENG and KAT
✅ Done! Created bilingual CSV: bible_eng_kat_perfects.csv
Rows: 440  (≈ 220 verses × 2)


,verse_id,language,text
0,40003002,ENG,"» Repent , for the kingdom of heaven has come ..."
1,40003002,KAT,"და იტყოდა : შეინანეთ , რამეთუ მოახლებულ არს სა..."
2,40003017,ENG,Then a voice from heaven said : » This is my S...
3,40003017,KAT,და ჴმაჲ იყო ზეცით და თქუა : ესე არს ძე ჩემი სა...
4,40005027,ENG,"» You have heard it was said , ‘ You should no..."
5,40005027,KAT,"გესმა , რამეთუ ითქუა : არა იმრუშო ."


In [ ]:
import pandas as pd
from pathlib import Path
import csv

# === Step 6: Build multilingual CSV of matching PLG verses ===

# Input files
plg_path = Path("plg_verses.csv")  # the list of PLG verse IDs
bible_files = {
    "ENG": Path("eng_bib.txt"),
    "DEU": Path("deu_bib.txt"),
    "CMN": Path("cmn_bib.txt"),
    "JPN": Path("jpn_bib.txt"),
}

# --- Load all Bibles into dictionaries ---
bibles = {lang: load_bible_txt_as_dict(path) for lang, path in bible_files.items()}
for lang, verses in bibles.items():
    print(f"{lang}: {len(verses)} verses loaded")

# --- Load list of verse IDs with PLGs ---
plg_ids = set(pd.read_csv(plg_path)["verse_id"].astype(str))
print(f"PLG verse IDs: {len(plg_ids)}")

# --- Keep only verses present in *all four* files ---
common_ids = plg_ids.copy()
for lang, verses in bibles.items():
    common_ids &= set(verses.keys())

print(f"✅ {len(common_ids)} verses found in ALL {len(bibles)} languages")

# --- Build DataFrame ---
records = []
for vid in sorted(common_ids):
    for lang, verses in bibles.items():
        records.append({
            "verse_id": vid,
            "lang": lang,
            "text": verses[vid],
        })

df_out = pd.DataFrame(records)

# --- Save output ---
output_path = Path("bible_multilingual_plg.csv")
df_out.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")

print(f"✅ Done! Created multilingual CSV: {output_path}")
print(f"Rows: {len(df_out)}  (≈ {len(common_ids)} verses × {len(bibles)})")
display(df_out.head(12))